# 患者筛选 + 元数据缓存（triage / meta_cache）

一次 Colab 跑完，解决三件事：

1. **筛选**：逐患者下载 → 真实 `preprocess` → 两道折过滤器 → 记录窗口数、折数、丢弃原因。
2. **元数据缓存**：每个 EDF 只记 `n_times / sfreq / ch_names`，存 `meta_cache/{pid}.json`。
   之后换 SOP / SPH / interictal_thr / 窗长 / 电极集合，都能在**本地**用这份缓存重算，不用再下载。
3. **仿真对拍**：用缓存模拟出窗口和折，和真实 `preprocess` 的结果逐项比对，确认仿真可信。

上传约定：把这一层文件夹整体传到 MyDrive 根目录，`cell-2` 的 `NAME` 改成它的名字。
算完立刻删 EDF，磁盘占用只按单患者峰值。

In [ ]:
!pip install -q mne

In [ ]:
import gc
import json
import sys
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

NAME = "261117_triage"                 # 上传到 MyDrive 根目录的文件夹名
CODE = Path("/content/drive/MyDrive") / NAME
DATA = Path("/content/chbmit")         # EDF 临时目录，算完删
OUT = CODE / "results"
META = CODE / "meta_cache"

assert CODE.exists(), "MyDrive 根目录下找不到 %s" % CODE
sys.path.insert(0, str(CODE))
DATA.mkdir(exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
META.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd

import tools as t
import colab_tools as ct

print("code =", CODE)

In [ ]:
ELECS = ["Fp1-F7", "Fp2-F8", "T7-P7", "T8-P8", "P7-O1", "P8-O2"]
PATIENTS = ["chb%02d" % i for i in range(1, 25)]

# 必须和训练时一致（仿真对拍用这组基准参数）
SPH, SOP = 2 * 60, 30 * 60
INTER_THR = 2 * 60 * 60
WIN_LEN, WIN_OL = 4, 2
ENC_VAL, CLS_VAL = 0.1, 0.1
SEED = 0
MIN_FOLDS = 3
FS_OUT = 200                        # 与 tools.FS_OUT 一致

CONFIG = dict(elecs=ELECS, sph=SPH, sop=SOP, interictal_thr=INTER_THR,
              window_length=WIN_LEN, window_overlap=WIN_OL,
              enc_val=ENC_VAL, cls_val=CLS_VAL, seed=SEED, min_folds=MIN_FOLDS)
CONFIG

## 第 0 步：只读 summary 的清单（不下载 EDF）

In [ ]:
inv = []
for pid in PATIENTS:
    ct.fetch("%s/%s-summary.txt" % (pid, pid), DATA)
    recs = t.read_summary(str(DATA / pid / ("%s-summary.txt" % pid)))
    inv.append(dict(patient=pid,
                    n_edf=len({r["file"] for r in recs}),
                    duration_h=round(sum(r["t1"] - r["t0"] for r in recs) / 3600.0, 2),
                    n_seizures=sum(len(r["seizures"]) for r in recs)))
inv = pd.DataFrame(inv)
inv.to_csv(OUT / "inventory.csv", index=False, encoding="utf-8")
inv

## 第 1 步：逐患者 下载 → 元数据 → preprocess → 折 → 结论

- 元数据先落盘（`meta_cache/{pid}.json`），后面任何参数扫描都靠它；
- 两道过滤器按训练顺序串联：`student_dataset`（preictal:间期）→ `teacher_from_student`（ictal:间期）；
- 算完立即删 EDF，可断点续跑（有 `{pid}.json` 就跳过）。

In [ ]:
def patient_meta(pdir, elecs):
    """只读头部：每个 EDF 的样本数、采样率、通道名（供离线仿真）"""
    import mne
    edf = {}
    for f in sorted(pdir.glob("*.edf")):
        try:
            raw = mne.io.read_raw_edf(str(f), preload=False, verbose="ERROR")
            names = [n.upper() for n in raw.ch_names]
            edf[f.name] = dict(n_times=int(raw.n_times), sfreq=float(raw.info["sfreq"]),
                               ch_names=names,
                               has_all=all(t._find(names, c) is not None for c in elecs))
        except Exception as e:
            edf[f.name] = dict(error=str(e), n_times=0, sfreq=float("nan"), ch_names=[], has_all=False)
    return edf


def triage(pid):
    pdir = ct.download_patient(pid, DATA)
    recs = t.read_summary(str(pdir / ("%s-summary.txt" % pid)))
    edf = patient_meta(pdir, ELECS)
    cache = dict(patient=pid, elecs=ELECS, summary=recs, edf=edf)
    with open(META / ("%s.json" % pid), "w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False)

    windows, meta = t.preprocess(str(pdir), SPH, SOP, INTER_THR, WIN_LEN, WIN_OL, ELECS)
    m0 = meta[ELECS[0]]
    labels = {int(k): int(v) for k, v in m0.label.value_counts().sort_index().items()} if len(m0) else {}
    n_pre = labels.get(1, 0)
    n_int = labels.get(0, 0)

    out = dict(patient=pid,
               n_edf=len(edf),
               n_edf_missing_channels=sum(1 for v in edf.values() if not v.get("has_all")),
               duration_h=round(sum(r["t1"] - r["t0"] for r in recs) / 3600.0, 2),
               n_seizures_summary=sum(len(r["seizures"]) for r in recs),
               n_windows=int(len(m0)),
               usable=int((m0.usable == 1).sum()) if len(m0) else 0,
               label_counts=labels,
               n_seizures_with_preictal=int(m0.loc[m0.label == 1, "seizure_id"].nunique()) if len(m0) else 0)

    if out["n_edf_missing_channels"]:
        out["missing_channels"] = sorted({c for v in edf.values() for c in ELECS
                                          if t._find([n.upper() for n in v.get("ch_names", [])], c) is None})

    # 没有 preictal 或没有间期窗时，split 本身跑不了（_split_loocv 要求有间期块）
    if n_pre == 0 or n_int == 0:
        out.update(folds_raw=0, folds_after_balance=0, folds_final=0, fold_detail_raw=[],
                   dropped_unbalanced=[], dropped_ictal=[], kept_seizures=[], min_train_windows=0)
        if out["n_windows"] == 0:
            why = "无可用窗口" + ("（缺 %s）" % ",".join(out.get("missing_channels", [])) if out.get("missing_channels") else "")
        elif n_pre == 0:
            why = "没有 preictal 窗（无标注发作）"
        else:
            why = "没有间期窗（所有窗都落在发作附近）"
        out["decision"], out["reason"] = "丢弃", why
        del windows, meta
        gc.collect()
        return out

    raw_folds = t.split_train_test(m0, seed=SEED, val=(ENC_VAL, CLS_VAL), method="LOOCV")
    out["fold_detail_raw"] = [dict(seizure_id=int(f["seizure_id"]),
                                   train=[f["n_train_pre"], f["n_train_inter"]],
                                   val=[f["n_val_pre"], f["n_val_inter"]],
                                   test=[f["n_test_pre"], f["n_test_inter"]]) for f in raw_folds]

    bal = ct.student_dataset(ELECS, meta, SEED, (ENC_VAL, CLS_VAL))[0]
    keep = {d["seizure_id"] for d in t.teacher_from_student(ELECS, meta, windows, bal, SEED)}
    final = [f for f in bal if f["seizure_id"] in keep]

    out["folds_raw"] = len(raw_folds)
    out["folds_after_balance"] = len(bal)
    out["folds_final"] = len(final)
    out["dropped_unbalanced"] = sorted(int(f["seizure_id"]) for f in raw_folds
                                       if not all(f["n_%s_pre" % k] == f["n_%s_inter" % k]
                                                  for k in ("train", "val", "test")))
    out["dropped_ictal"] = sorted(int(f["seizure_id"]) for f in bal if f["seizure_id"] not in keep)
    out["kept_seizures"] = sorted(int(f["seizure_id"]) for f in final)
    out["min_train_windows"] = int(min(f["n_train_pre"] + f["n_train_inter"] for f in final)) if final else 0

    if out["folds_final"] == 0:
        decision, why = "丢弃", "没有可用折"
    elif out["folds_final"] < MIN_FOLDS:
        decision, why = "丢弃", "最终折数 %d < %d" % (out["folds_final"], MIN_FOLDS)
    else:
        decision, why = "保留", ""
    out["decision"], out["reason"] = decision, why

    del windows, meta
    gc.collect()
    return out

In [ ]:
for pid in PATIENTS:
    path = OUT / ("%s.json" % pid)
    if path.exists():
        print("[跳过]", pid)
        continue
    print("=" * 15, pid, "=" * 15)
    try:
        rec = triage(pid)
    except Exception as e:
        print("[失败] %s: %r" % (pid, e))
        ct.clear_patient(pid, DATA)
        continue
    with open(path, "w", encoding="utf-8") as f:
        json.dump(rec, f, indent=4, ensure_ascii=False)
    ct.clear_patient(pid, DATA)
    print("[%s] 窗 %d / 可用 %d | 折 %d → %d → %d | %s %s"
          % (pid, rec["n_windows"], rec["usable"], rec["folds_raw"],
             rec["folds_after_balance"], rec["folds_final"], rec["decision"], rec["reason"]))
    gc.collect()

## 第 2 步：用 meta_cache 仿真，和真实结果对拍

仿真只用 `n_times / sfreq / ch_names` + summary 的发作时刻，重建每一个窗的标签；
窗口数、label 计数、折表必须和上面的真实结果一致，否则不能用它做离线扫参。

In [ ]:
def simulate(cache, SPH, SOP, INTER_THR, WIN_LEN, WIN_OL, elecs, enc_val, cls_val, seed):
    """用元数据重建窗口表 + 折；不读任何信号"""
    rows = []
    W = int(round(WIN_LEN * FS_OUT))
    O = int(round(WIN_OL * FS_OUT))
    step = W - O
    seize = sorted(s for rec in cache["summary"] for s in rec["seizures"])   # 全局发作表（同 preprocess）
    for rec in cache["summary"]:
        info = cache["edf"].get(rec["file"])
        if info is None or info.get("error"):
            continue
        names = [n.upper() for n in info["ch_names"]]
        if any(t._find(names, c) is None for c in elecs):
            continue
        L = -(-int(info["n_times"]) * FS_OUT // int(round(info["sfreq"])))
        if step <= 0 or L < W:
            continue
        n_win = (L - W) // step + 1
        for k in range(n_win):
            s = k * step
            d = t.label_window(rec["t0"] + s / FS_OUT, rec["t0"] + (s + W) / FS_OUT,
                               seize, SPH, SOP, INTER_THR)
            rows.append(dict(label=d["label"], seizure_id=d["seizure_id"],
                             prev_seizure_id=d["prev_seizure_id"],
                             next_seizure_id=d["next_seizure_id"],
                             center_abs_s=rec["t0"] + (s + W / 2.0) / FS_OUT,
                             usable=d["usable"], excluded_reason=d["excluded_reason"]))
    meta = pd.DataFrame(rows)
    if not len(meta):
        return dict(n_windows=0, labels={}, folds_raw=0, folds_after_balance=0, folds_final=0)
    labels = {int(k): int(v) for k, v in meta.label.value_counts().sort_index().items()}
    # 和 triage 一样：没有 preictal 或没有间期窗时 split 跑不了
    if labels.get(1, 0) == 0 or labels.get(0, 0) == 0:
        return dict(n_windows=int(len(meta)), labels=labels,
                    folds_raw=0, folds_after_balance=0, folds_final=0)

    md = {e: meta for e in elecs}
    dummy = {e: np.zeros(len(meta), dtype=np.float32) for e in elecs}
    raw_folds = t.split_train_test(meta, seed=seed, val=(enc_val, cls_val), method="LOOCV")
    bal = ct.student_dataset(elecs, md, seed, (enc_val, cls_val))[0]
    keep = {d["seizure_id"] for d in t.teacher_from_student(elecs, md, dummy, bal, seed)}
    return dict(n_windows=int(len(meta)), labels=labels,
                folds_raw=len(raw_folds), folds_after_balance=len(bal),
                folds_final=sum(1 for f in bal if f["seizure_id"] in keep))

In [ ]:
bad = []
for pid in PATIENTS:
    mp, rp = META / ("%s.json" % pid), OUT / ("%s.json" % pid)
    if not (mp.exists() and rp.exists()):
        continue
    cache = json.loads(mp.read_text(encoding="utf-8"))
    real = json.loads(rp.read_text(encoding="utf-8"))
    sim = simulate(cache, SPH, SOP, INTER_THR, WIN_LEN, WIN_OL, ELECS, ENC_VAL, CLS_VAL, SEED)
    ok = (sim["n_windows"] == real["n_windows"] and sim["labels"] == real["label_counts"]
          and sim["folds_raw"] == real["folds_raw"]
          and sim["folds_after_balance"] == real["folds_after_balance"]
          and sim["folds_final"] == real["folds_final"])
    print("%s 窗 %d/%d 折 %d-%d-%d/%d-%d-%d %s"
          % (pid, sim["n_windows"], real["n_windows"],
             sim["folds_raw"], sim["folds_after_balance"], sim["folds_final"],
             real["folds_raw"], real["folds_after_balance"], real["folds_final"],
             "OK" if ok else "MISMATCH"))
    if not ok:
        bad.append(pid)
print("不一致:", bad or "无")

## 第 3 步：汇总筛选表

In [ ]:
rows = []
for p in sorted(OUT.glob("chb*.json")):
    r = json.loads(p.read_text(encoding="utf-8"))
    rows.append(dict(patient=r["patient"], n_edf=r["n_edf"],
                     bad_edf=r.get("n_edf_missing_channels", 0),
                     duration_h=r["duration_h"], n_seizures=r["n_seizures_summary"],
                     windows=r["n_windows"], usable=r["usable"],
                     folds_raw=r["folds_raw"], folds_after_balance=r["folds_after_balance"],
                     folds_final=r["folds_final"], min_train_windows=r["min_train_windows"],
                     decision=r["decision"], reason=r["reason"]))
df = pd.DataFrame(rows).sort_values("patient").reset_index(drop=True)
df.to_csv(OUT / "triage.csv", index=False, encoding="utf-8")
print("保留 %d / 丢弃 %d" % ((df.decision == "保留").sum(), (df.decision == "丢弃").sum()))
df